In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, classification_report
)

In [3]:
df = pd.read_csv("student-data.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nMissing values in each column:")
print(df.isnull().sum())

Dataset shape: (395, 31)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,passed
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,no,no,4,3,4,1,1,3,6,no
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,yes,no,5,3,3,1,1,3,4,no
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,yes,no,4,3,2,2,3,3,10,yes
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,yes,3,2,2,1,1,5,2,yes
4,GP,F,16,U,GT3,T,3,3,other,other,...,no,no,4,3,2,1,2,5,4,yes


\nColumn names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'passed']
\nMissing values in each column:
school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
passed        0
dtype: int64


In [4]:
print("Pass/fail counts:")
print(df["passed"].value_counts())

print("\\nAbsences summary:")
print(df["absences"].describe())

Pass/fail counts:
passed
yes    265
no     130
Name: count, dtype: int64
\nAbsences summary:
count    395.000000
mean       5.708861
std        8.003096
min        0.000000
25%        0.000000
50%        4.000000
75%        8.000000
max       75.000000
Name: absences, dtype: float64


In [6]:
X_lin = df.drop(columns=["absences", "passed"])
y_lin = df["absences"]

print("Input shape:", X_lin.shape)
print("Output shape:", y_lin.shape)

Input shape: (395, 29)
Output shape: (395,)


In [7]:
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_lin, y_lin,
    test_size=0.20,
    random_state=42
)

print("Training records:", len(X_train_lin))
print("Testing records:", len(X_test_lin))

Training records: 316
Testing records: 79


In [10]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
numeric_features_lin = X_lin.select_dtypes(include=["number"]).columns.tolist()
categorical_features_lin = X_lin.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor_lin = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features_lin),
    ("cat", categorical_transformer, categorical_features_lin)
])

lin_model = Pipeline(steps=[
    ("preprocessor", preprocessor_lin),
    ("regressor", LinearRegression())
])

lin_model.fit(X_train_lin, y_train_lin)
print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [11]:
y_pred_lin = lin_model.predict(X_test_lin)

print("Actual absences:", y_test_lin.to_numpy())
print("Predicted absences:", np.round(y_pred_lin, 2))

mae = mean_absolute_error(y_test_lin, y_pred_lin)
mse = mean_squared_error(y_test_lin, y_pred_lin)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_lin, y_pred_lin)

print("\\nMAE:", round(mae, 3))
print("MSE:", round(mse, 3))
print("RMSE:", round(rmse, 3))
print("R2 Score:", round(r2, 3))

Actual absences: [ 2  3  8  8 11  0  2 18  0  2  0  6  1  0  8  2  0  2  0  0  4  3  0  6
  0 10  0  0  0  8 38  2  0  4  0 14  0  4  2 14  2  0  4  8  2  0 12 20
  6  4  8  0  0  4  7 12  2  0  2  0  6  8  6  4  4  0  2  4  9 10  8  2
  9  6  0 10  8  2 16]
Predicted absences: [ 9.33  4.69  9.61  7.15  3.7  12.72 -2.06  2.02  6.16 -2.98  3.95  8.76
  6.4   8.57  7.81 -5.4   6.42  4.11  4.39  0.85  7.89  1.96  6.24  6.28
 -2.22 -0.57  7.49  1.08  0.2  -2.71  5.76  8.3   7.66  2.9  11.25  4.53
 11.03 -0.5   4.4   9.49 10.4   0.82  5.21  0.72  1.15  6.9  14.09  8.94
  8.58  8.12  4.33  8.55  7.44  7.69  8.46  7.34  0.49  5.52  4.67  4.59
 -2.1  11.28  7.29  6.69  6.74  4.97 -1.42  4.2   8.09  4.91 11.28  8.04
  4.52  0.61 -0.72  2.24  2.16  7.7   4.09]
\nMAE: 5.212
MSE: 48.213
RMSE: 6.944
R2 Score: -0.345
